# Neural Network Month · Project 2
## Notebook 1 of 3 — Cleaning the Mess

Project 1 taught the mechanics on tidy synthetic data. Project 2 meets the real thing: **a public brain-tumor MRI collection**, with a convolutional neural network (CNN) classifying scans by tumor type.

Our philosophy for this project: **change the data as little as possible, and make the model strong enough to cope with what is left.** So this notebook does only the cleaning that is *necessary*, and shows its reasoning for each step.

**The dataset:** *MRI for Brain Tumor with Bounding Boxes* (Kaggle). The listing describes 5,249 MRI images across four classes (glioma, meningioma, pituitary, no tumor), with a ready-made train/validation split. We use only the class folders as labels and ignore the bounding boxes.

**What we will do**

1. Keep only the **512×512** scans, so nothing ever has to be resized.
2. Remove **exact duplicate files**, because a duplicate that lands on both sides of a train/test split lets a model "pass" by memorising.
3. Replace the shipped split with **stratified 5-fold cross-validation**.

Four figures.

## 1 · Setup and palette

In [ ]:
import os, glob, zipfile, hashlib, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from pathlib import Path
from PIL import Image
from sklearn.model_selection import StratifiedKFold

# One palette for the whole series. Each tumor class keeps its colour in every figure.
P = {
    "glioma":     "#C97B4A",  # clay
    "meningioma": "#2F6F73",  # teal
    "pituitary":  "#D9A441",  # ochre
    "no_tumor":   "#7A9E7E",  # sage
    "ink":        "#4A3F35",  # bark   - text, validation lines
    "paper":      "#F5EFE6",  # sand   - background
    "grid":       "#D8CFC0",  # stone  - gridlines
    "stone":      "#A89F91",  # neutral bars and training lines
    "rust":       "#A5402D",  # warning colour (duplicates, errors)
}
CLASS_COLORS = {"Glioma": P["glioma"], "Meningioma": P["meningioma"],
                "Pituitary": P["pituitary"], "No Tumor": P["no_tumor"]}
SEQ_CMAP = LinearSegmentedColormap.from_list("seq", [P["paper"], P["ink"]])
CAM_CMAP = LinearSegmentedColormap.from_list("cam", [P["pituitary"], P["glioma"], P["rust"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

## 2 · Get the data

Download `archive.zip` from the dataset's Kaggle page, upload it to Colab (left panel → Files), and run the next cell. If you saved it elsewhere, change `ZIP_PATH`.

In [ ]:
# Where is the data? The cell tries, in order: an already-extracted folder, your uploaded
# Kaggle zip, then a Kaggle download. Easiest route: download archive.zip from the Kaggle page
# and upload it to /content (the Colab file panel), or point ZIP_PATH at it in Google Drive.
ZIP_PATH  = "/content/archive.zip"
DATA_DIR  = "mri_data"

def locate_root(base):
    for p in glob.glob(f"{base}/**/Train", recursive=True):
        return os.path.dirname(p)
    return None

root = locate_root(DATA_DIR) if os.path.isdir(DATA_DIR) else None
if root is None and os.path.exists(ZIP_PATH):
    zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
    root = locate_root(DATA_DIR)
if root is None:
    try:
        import kagglehub
        path = kagglehub.dataset_download("ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes")
        root = locate_root(path)
    except Exception as e:
        print("Kaggle download did not work:", e)
if root is None:
    raise FileNotFoundError("Upload the Kaggle zip to Colab and set ZIP_PATH, or add Kaggle credentials.")
print("Data folder:", root)

## 3 · Index every image

For each file we record where it is, which shipped split it was in, its class (the folder name), its pixel dimensions, and an **MD5 fingerprint** of its bytes. Two files with the same fingerprint are byte-for-byte identical.

In [ ]:
# ---- Cleaning settings (the only places we touch the data) ----
KEEP_ONLY_512   = True   # keep only 512x512 scans, so nothing is ever resized
MIN_CLASS_COUNT = 100    # a class with fewer images than this after cleaning is dropped
N_FOLDS, SEED   = 5, 42

def index_images(root):
    """One row per image: where it is, its shipped split, class, size, and a fingerprint of its bytes."""
    rows = []
    for p in sorted(glob.glob(f"{root}/*/*/images/*.jpg")):
        rel = os.path.relpath(p, root)
        split, cls = rel.split(os.sep)[:2]
        with Image.open(p) as im:
            w, h = im.size
        with open(p, "rb") as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        rows.append((rel, split, cls, w, h, md5))
    return pd.DataFrame(rows, columns=["path", "shipped_split", "cls", "w", "h", "md5"])

def clean(raw):
    stages = {"Raw Download": raw}
    df = raw
    if KEEP_ONLY_512:
        df = df[(df.w == 512) & (df.h == 512)]
    stages["512 Pixels Only" if KEEP_ONLY_512 else "All Sizes Kept"] = df

    # How contaminated is the shipped validation set? (measured before removing anything else)
    train_hashes = set(df[df.shipped_split == "Train"].md5)
    val = df[df.shipped_split == "Val"]
    leak = {"val_total": len(val), "val_copies": int(val.md5.isin(train_hashes).sum()),
            "all_total": len(df), "all_redundant": int(len(df) - df.md5.nunique())}

    # Drop any file whose identical twin carries a different label, then keep one copy of each image
    mixed = df.groupby("md5")["cls"].nunique()
    leak["conflicting_labels"] = int((mixed > 1).sum())
    df = df[~df.md5.isin(mixed[mixed > 1].index)]
    df = df.sort_values(["shipped_split", "path"]).drop_duplicates("md5", keep="first")
    stages["Duplicates Removed"] = df

    counts = df.cls.value_counts()
    dropped = {c: int(n) for c, n in counts.items() if n < MIN_CLASS_COUNT}
    df = df[~df.cls.isin(list(dropped))]
    stages["Final Dataset"] = df

    df = df.reset_index(drop=True)
    classes = sorted(df.cls.unique())
    df["label"] = df.cls.map({c: i for i, c in enumerate(classes)})
    df["fold"] = -1
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for k, (_, test_idx) in enumerate(skf.split(df, df.label)):
        df.loc[test_idx, "fold"] = k
    return df, stages, leak, dropped, classes

In [ ]:
raw = index_images(root)
print(f"{len(raw)} images indexed")
raw.groupby(["cls", "shipped_split"]).size().unstack()

## 4 · Clean, and count what survives each step

Each stage is kept so we can plot it. The last stage drops any class left with fewer than `MIN_CLASS_COUNT` images, because a class with a few dozen examples cannot be learned or validated.

In [ ]:
manifest, stages, leak, dropped, classes = clean(raw)

summary = pd.DataFrame({name: d.cls.value_counts() for name, d in stages.items()}).fillna(0).astype(int)
summary.loc["Total"] = summary.sum()
print("Images per class at each stage:")
display(summary)
if dropped:
    print(f"\nClass dropped for having under {MIN_CLASS_COUNT} images after cleaning: {dropped}")
print(f"\nFinal dataset: {len(manifest)} scans in {len(classes)} classes: {classes}")

## 5 · Figure 1 — what each cleaning step does

In [ ]:
order = sorted(raw.cls.unique())
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4), sharey=True)
for ax, (name, d) in zip(axes, stages.items()):
    counts = d.cls.value_counts().reindex(order, fill_value=0)
    bars = ax.bar(range(len(order)), counts.values, color=[CLASS_COLORS[c] for c in order])
    ax.bar_label(bars, padding=2, fontsize=9)
    ax.set_xticks(range(len(order))); ax.set_xticklabels([c.replace(" ", "\n") for c in order], fontsize=9)
    ax.set_title(name)
axes[0].set_ylabel("Images")
fig.suptitle("Class Counts by Stage", y=1.04)
fig.tight_layout(); save_fig(fig, "01_class_counts_by_stage"); plt.show()

**The key finding is in the second panel.** Almost all of the *No Tumor* scans are not 512×512, so keeping only 512×512 leaves just a handful. That is too few to learn or to validate across five folds, so in the default settings the pipeline drops the class and this becomes a **three-way tumor-type problem: glioma vs. meningioma vs. pituitary.**

That is a real trade-off, and a defensible one. The no-tumor scans also differ from the tumor scans in more than the tumor (in a quick look they were mostly axial slices with a different contrast, while many tumor scans are contrast-enhanced and viewed from other angles), so a network could "detect" them from image style alone. Telling tumor *types* apart is the harder and more honest question.

To keep *No Tumor* instead, set `KEEP_ONLY_512 = False` in the cleaning cell and re-run. Every scan is then resized to 512×512 when loaded, and you accept that resizing, and the style difference, as part of the data.

## 6 · Figure 2 — a look at the scans

In [ ]:
rng = np.random.default_rng(3)
fig, axes = plt.subplots(len(classes), 5, figsize=(12.5, 2.55 * len(classes)))
axes = np.atleast_2d(axes)
for r, c in enumerate(classes):
    pick = rng.choice(manifest.index[manifest.cls == c], 5, replace=False)
    for a, i in zip(axes[r], pick):
        a.imshow(Image.open(os.path.join(root, manifest.path[i])).convert("L"), cmap="gray"); a.axis("off")
    axes[r][0].text(-0.06, 0.5, c, transform=axes[r][0].transAxes, rotation=90, ha="right", va="center",
                    fontsize=13, fontweight="bold", color=CLASS_COLORS[c])
fig.suptitle("Sample Scans per Class", y=1.0)
fig.tight_layout(); save_fig(fig, "02_sample_scans"); plt.show()

Notice how varied these are: different angles (sagittal, coronal, axial), different zoom, different brightness. We are **not** normalising any of it. That variety is exactly what the network has to learn to see past.

## 7 · Figure 3 — why the shipped split cannot be trusted

In [ ]:
scope = "512 images" if KEEP_ONLY_512 else "All images"
bars = [("Shipped validation\nset", leak["val_copies"], leak["val_total"]),
        (f"{scope},\nwhole collection", leak["all_redundant"], leak["all_total"])]
fig, ax = plt.subplots(figsize=(10, 3.8))
for y, (label, dup, tot) in enumerate(bars):
    ax.barh(y, dup / tot, color=P["rust"], height=0.55, label="Exact copy" if y == 0 else None)
    ax.barh(y, 1 - dup / tot, left=dup / tot, color=P["stone"], height=0.55, label="Unique" if y == 0 else None)
    ax.text(dup / tot / 2, y, f"{dup/tot:.0%}", ha="center", va="center", color="white", fontweight="bold", fontsize=13)
    ax.text(dup / tot + (1 - dup / tot) / 2, y, f"{1 - dup/tot:.0%}", ha="center", va="center", color="white", fontweight="bold", fontsize=13)
    ax.text(1.01, y, f"{dup:,} of {tot:,}", va="center", fontsize=10)
ax.set_yticks(range(len(bars))); ax.set_yticklabels([b[0] for b in bars]); ax.invert_yaxis()
ax.set_xlim(0, 1.18); ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0]); ax.set_xticklabels(["0%", "25%", "50%", "75%", "100%"])
ax.set_xlabel("Share of images")
ax.legend(frameon=False, ncol=2, loc="upper center", bbox_to_anchor=(0.45, -0.22))
ax.set_title("Validation Leakage Found")
fig.tight_layout(); save_fig(fig, "03_validation_leakage"); plt.show()
print(f"Shipped validation images that are byte-identical to a training image: {leak['val_copies']} of {leak['val_total']}")

A large share of the shipped validation images are **exact copies of training images**. A network scored on that split would be partly graded on images it has already seen, which inflates the score. Removing duplicate files fixes the exact copies; it cannot fix everything (see the limits at the end).

Next we replace the shipped split with **stratified 5-fold cross-validation**: the scans are divided into five groups, each with the same class mix, and each group takes a turn as the held-out test set.

## 8 · Figure 4 — are the folds balanced?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
bottom = np.zeros(N_FOLDS)
for c in classes:
    n = np.array([((manifest.fold == k) & (manifest.cls == c)).sum() for k in range(N_FOLDS)])
    ax.bar(range(N_FOLDS), n, bottom=bottom, color=CLASS_COLORS[c], label=c, width=0.65)
    for k in range(N_FOLDS):
        ax.text(k, bottom[k] + n[k] / 2, str(n[k]), ha="center", va="center", color="white", fontsize=10, fontweight="bold")
    bottom += n
for k in range(N_FOLDS):
    ax.text(k, bottom[k] + 8, str(int(bottom[k])), ha="center", fontsize=10)
ax.set_xticks(range(N_FOLDS)); ax.set_xticklabels([f"Fold {k+1}" for k in range(N_FOLDS)])
ax.set_ylabel("Scans"); ax.legend(frameon=False, ncol=len(classes), loc="upper center", bbox_to_anchor=(0.5, -0.1))
ax.set_title("Stratified Fold Composition")
fig.tight_layout(); save_fig(fig, "04_fold_composition"); plt.show()

In [ ]:
manifest.to_csv("clean_manifest.csv", index=False)
print("Saved clean_manifest.csv (scan list, labels and fold assignments) for Notebooks 2 and 3.")

## What we did, and what we did not

**Changed:** kept 512×512 scans only; removed exact-duplicate files; replaced the shipped split with stratified folds.
**Not changed:** no resizing, no cropping, no intensity normalisation, no hand-picked images.

**Limits you should state in the newsletter**

- We removed *exact* duplicates only. Near-copies (the same patient's neighbouring slices, or slightly edited versions of one image) can still sit on both sides of a fold, so cross-validated scores may be **optimistic**. The files carry no patient IDs, so we cannot rule this out.
- The collection is a compilation of public sources, and the *"cleaned and curated"* claim is the uploader's, not something we verified. Credit the original sources alongside the Kaggle page.
- Scan angle and contrast differ between classes, so a network may partly learn those instead of the tumor. Notebook 3 checks where the network looks.

**Next:** Notebook 2 builds the CNN and shows what it does to a scan, layer by layer.

In [ ]:
# Optional: download this notebook's figures and the manifest
try:
    import shutil
    from google.colab import files
    shutil.make_archive("nb1_figures", "zip", "figures")
    files.download("nb1_figures.zip"); files.download("clean_manifest.csv")
except ImportError:
    print("Figures are saved in the 'figures' folder.")